# COSC 661 · Combined HW3/4 — STUDENT CODING STARTER

**Seven required coding tasks** are marked `TODO 1` through `TODO 7`. Replace every `raise NotImplementedError` and complete the requested functions. Run cells in order on a Colab GPU. Upload `train.csv`, `test.csv`, and `evaluation.py` to `/content/`. Do not submit an unmodified starter. Human scores are assigned manually, not generated automatically.


In [ ]:
!pip -q install "transformers>=4.45,<5" "peft>=0.13,<1" "accelerate>=1,<2" pandas

In [ ]:
import os, sys, time, random, numpy as np, pandas as pd, torch, transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model
from evaluation import format_compliant, summarize_outputs, validate_results, write_csv
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
print("torch",torch.__version__,"transformers",transformers.__version__,"peft",peft.__version__)
print("GPU",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")
assert torch.cuda.is_available(), "This LoRA lab requires a GPU runtime."

In [ ]:
train_df=pd.read_csv("/content/train.csv"); test_df=pd.read_csv("/content/test.csv")
assert len(train_df)==160 and len(test_df)==20
assert not(set(train_df.id)&set(test_df.id))
print(train_df.head().to_string(index=False));print("Train/test:",len(train_df),len(test_df))

## Model loading
**Recommended:** `Qwen/Qwen2.5-0.5B-Instruct`. The model is downloaded from Hugging Face; no API key is required for public access. A Colab T4 or better is recommended.

In [ ]:
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to("cuda")
model.config.use_cache=False
print("Parameters:",sum(p.numel() for p in model.parameters()))

## Part A — Student coding tasks 1 and 2
Use only training examples for few-shot demonstrations. Keep the same model for all conditions.


In [ ]:
examples=train_df.head(3).to_dict("records")

# TODO 1: Implement both prompt builders.
def basic_prompt(text):
    """Return a zero-shot instruction with the input and a clear output request."""
    raise NotImplementedError("TODO 1: basic_prompt")

def improved_prompt(text):
    """Use the 3 training examples, preserve meaning, require professional tone,
    forbid added information, and request only the rewritten sentence."""
    raise NotImplementedError("TODO 1: improved_prompt")

# TODO 2: Implement inference with two strategies: greedy and top-p.
def generate(model, prompt, strategy="greedy"):
    """Render a Qwen chat prompt, tokenize, generate and decode ONLY new tokens.
    greedy: do_sample=False; top_p: do_sample=True, temperature=.7, top_p=.9.
    Both: max_new_tokens=65; pad_token_id=tokenizer.eos_token_id.
    """
    raise NotImplementedError("TODO 2: generate")


In [ ]:
# TODO 2 CHECK: run only after implementing prompt functions and generate().
assert isinstance(basic_prompt("Send it."),str) and "Send it." in basic_prompt("Send it.")
assert "Send it." in improved_prompt("Send it.")

# Instructor-provided experiment runner: 5 shared inputs, two decoding settings.
decoding_rows=[]
for r in test_df.head(5).to_dict("records"):
    for strategy in ("greedy","top_p"):
        decoding_rows.append({"id":r["id"],"strategy":strategy,"input":r["input"],
                              "output":generate(model,improved_prompt(r["input"]),strategy)})
pd.DataFrame(decoding_rows).to_csv("decoding_comparison.csv",index=False)
display(pd.DataFrame(decoding_rows))


## Part B — Student coding tasks 3 and 4
Implement response-only loss labels: prompt positions must be `-100`. The provided collator and Trainer handle padding and optimization.


In [ ]:
MAX_LENGTH=192

# TODO 3: implement example formatting + tokenization.
def encode_training_example(row):
    """Return dict with input_ids, attention_mask, labels.
    1. Render a user message using basic_prompt(row['input']) with chat template.
    2. Tokenize prefix and target response + eos separately (no extra special tokens).
    3. Concatenate and truncate to MAX_LENGTH.
    4. Mask prefix label positions with -100; leave target IDs as labels.
    5. Assert at least one target label survives.
    """
    raise NotImplementedError("TODO 3: encode_training_example")

class RewritingDataset(torch.utils.data.Dataset):
    def __init__(self,df): self.rows=df.to_dict("records")
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return encode_training_example(self.rows[i])

def collate(features):
    max_len=max(len(f["input_ids"]) for f in features)
    batch={}
    for key in ("input_ids","attention_mask","labels"):
        pad_value=-100 if key=="labels" else (tokenizer.pad_token_id if key=="input_ids" else 0)
        batch[key]=torch.tensor([f[key]+[pad_value]*(max_len-len(f[key])) for f in features],dtype=torch.long)
    return batch

training_dataset=RewritingDataset(train_df)
example=training_dataset[0]
assert len(example['input_ids'])==len(example['labels'])==len(example['attention_mask'])
assert -100 in example['labels'] and any(x!=-100 for x in example['labels'])
print('Tokenized example length:',len(example['input_ids']))


In [ ]:
# TODO 4: Configure LoRA for Qwen2.5-0.5B-Instruct.
# Requirements: CAUSAL_LM, r=8, alpha=16, dropout=.05, target q_proj/v_proj.
# Assign your LoraConfig object to `config`.
raise NotImplementedError("TODO 4: assign config = LoraConfig(...)")

model=get_peft_model(model,config)
model.print_trainable_parameters()
model.config.use_cache=False
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
total=sum(p.numel() for p in model.parameters())
print(f'Trainable percentage: {100*trainable/total:.3f}%')


In [ ]:
args=TrainingArguments(output_dir="/content/hw34_checkpoints",num_train_epochs=1,per_device_train_batch_size=2,gradient_accumulation_steps=4,learning_rate=2e-4,logging_steps=5,save_strategy="no",report_to="none",fp16=True,remove_unused_columns=False,seed=SEED)
trainer=Trainer(model=model,args=args,train_dataset=training_dataset,data_collator=collate)
t0=time.time();train_output=trainer.train();print("Training seconds",round(time.time()-t0,1),"train loss",train_output.training_loss)
model.save_pretrained("/content/hw34_lora_adapter");tokenizer.save_pretrained("/content/hw34_lora_adapter")

## Part C — Student coding tasks 5 and 6
**Controlled comparison:** A=base/basic/greedy, B=base/few-shot/greedy, C=LoRA/basic/greedy. The runner unloads LoRA before loading the base model to conserve GPU memory.


In [ ]:
# TODO 5: Implement the comparison runner. Return one record per (id, condition).
def evaluate_condition(model, df, condition, prompt_fn):
    """Return list of dicts with keys id, condition, input, output.
    Generate with strategy='greedy' for every record.
    """
    raise NotImplementedError("TODO 5: evaluate_condition")

results=evaluate_condition(model,test_df,'C',basic_prompt)
del trainer,model
import gc
gc.collect();torch.cuda.empty_cache()
base_model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to('cuda')
results += evaluate_condition(base_model,test_df,'A',basic_prompt)
results += evaluate_condition(base_model,test_df,'B',improved_prompt)
validate_results(results,expected_ids=test_df.id.tolist())
write_csv('results.csv',results,['id','condition','input','output'])
print('Saved results.csv')

# TODO 6: calculate format compliance by condition using format_compliant(text).
# Return a DataFrame with columns: condition, n, compliant, compliance_rate.
def compliance_summary(rows):
    raise NotImplementedError("TODO 6: compliance_summary")

display(compliance_summary(results))


## Part D — Student coding task 7: human ratings and analysis
The scores **must be entered by a human**. Generate the blank sheet once, download it, enter integer scores 1–5 for meaning, tone and fluency, and upload it as `human_ratings_completed.csv`. **Do not rerun the blank-sheet creation after entering scores.**


In [ ]:
selected_ids=test_df.id.head(10).tolist()
ratings=[{'id':r['id'],'condition':r['condition'],'input':r['input'],'output':r['output'],
          'meaning_1to5':'','tone_1to5':'','fluency_1to5':'','notes':''}
         for r in results if r['id'] in selected_ids]
write_csv('human_ratings.csv',ratings,
          ['id','condition','input','output','meaning_1to5','tone_1to5','fluency_1to5','notes'])
print('Blank human_ratings.csv created: 30 rows. Download, rate manually, re-upload.')


## TODO 7 — Human evaluation and failure analysis

After manually rating 10 shared test inputs for each of A, B and C:

1. Upload `human_ratings_completed.csv` to Colab.
2. Implement `validate_and_summarize_ratings(path)` below. Reject missing/duplicate/unexpected `(id, condition)` pairs and non-integer ratings outside 1–5. Return average meaning, tone, fluency and overall scores per condition.
3. Identify **three failure cases** using `results.csv`, with the input, model output, and your explanation. Put this analysis in the report.

The next code cell contains a required TODO; it should not automatically fabricate human scores.


In [ ]:
# TODO 7: implement validation and summary of manually completed ratings.
RATING_COLS=['meaning_1to5','tone_1to5','fluency_1to5']

def validate_and_summarize_ratings(path='human_ratings_completed.csv'):
    """Read the completed CSV; validate 30 unique (id, condition) keys
    against the generated blank sheet; require integer 1..5 in all rating
    columns; return a DataFrame indexed by condition with mean rating columns
    and an 'overall' column (mean of the three criterion means).
    Raise ValueError on incomplete or invalid files.
    """
    raise NotImplementedError('TODO 7: validate_and_summarize_ratings')

# Uncomment after uploading the completed file:
# rating_summary=validate_and_summarize_ratings()
# display(rating_summary)
# rating_summary.to_csv('human_evaluation_summary.csv')


## Final submission checklist
- [ ] TODO 1–7 implemented and run without `NotImplementedError`.
- [ ] `decoding_comparison.csv` (5 IDs × 2 strategies).
- [ ] `results.csv` (20 IDs × 3 conditions).
- [ ] `human_ratings_completed.csv` (30 manually rated responses).
- [ ] `human_evaluation_summary.csv` (averages by condition).
- [ ] Saved LoRA adapter and 4–5 page report with three failure cases.

**Grading reminder:** Human ratings are not generated by code; the validation function checks human-entered scores. Do not use test references as few-shot examples or training samples.
